In [6]:
!pip install -q streamlit plotly pandas numpy scikit-learn pyngrok

In [2]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧
up to date, audited 23 packages in 1s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠇

In [7]:
%%writefile app.py
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

st.set_page_config(layout="wide", page_title="Dashboard Cianobacterias")

st.title("Monitoreo de Cianobacterias - Lagos Amatitlán y Atitlán")

np.random.seed(42)
n_samples = 500
data = pd.DataFrame({
    'lago': np.random.choice(['Amatitlán', 'Atitlán'], n_samples),
    'concentracion': np.random.exponential(50, n_samples),
    'temperatura': np.random.normal(22, 3, n_samples),
    'ph': np.random.normal(7.5, 0.8, n_samples),
    'oxigeno': np.random.normal(6, 1.5, n_samples),
    'mes': np.random.choice(range(1, 13), n_samples),
    'profundidad': np.random.uniform(0, 50, n_samples)
})
data['nivel_riesgo'] = pd.cut(data['concentracion'], bins=[0, 30, 60, 200], labels=['Bajo', 'Medio', 'Alto'])

data = data.dropna()

st.sidebar.header("Filtros")
lago_seleccionado = st.sidebar.multiselect("Seleccionar Lagos", options=['Amatitlán', 'Atitlán'], default=['Amatitlán', 'Atitlán'])
mes_seleccionado = st.sidebar.slider("Rango de Meses", 1, 12, (1, 12))

data_filtrada = data[data['lago'].isin(lago_seleccionado) & data['mes'].between(mes_seleccionado[0], mes_seleccionado[1])]

col1, col2 = st.columns(2)

with col1:
    st.subheader("Distribución de Concentración por Lago")
    fig1 = px.box(data_filtrada, x='lago', y='concentracion', color='lago',
                  color_discrete_map={'Amatitlán': '#2E5090', 'Atitlán': '#4A7BB7'})
    fig1.update_layout(showlegend=False)
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("Niveles de Riesgo")
    riesgo_counts = data_filtrada['nivel_riesgo'].value_counts()
    fig2 = px.pie(values=riesgo_counts.values, names=riesgo_counts.index,
                  color_discrete_sequence=['#2E5090', '#FF8C42', '#D64545'])
    st.plotly_chart(fig2, use_container_width=True)

col3, col4 = st.columns(2)

with col3:
    st.subheader("Concentración vs Temperatura")
    fig3 = px.scatter(data_filtrada, x='temperatura', y='concentracion', color='lago',
                      size='oxigeno', hover_data=['ph', 'profundidad'],
                      color_discrete_map={'Amatitlán': '#2E5090', 'Atitlán': '#FF8C42'})
    st.plotly_chart(fig3, use_container_width=True)

with col4:
    st.subheader("Variación Mensual")
    mes_agg = data_filtrada.groupby(['mes', 'lago'])['concentracion'].mean().reset_index()
    fig4 = px.line(mes_agg, x='mes', y='concentracion', color='lago',
                   color_discrete_map={'Amatitlán': '#2E5090', 'Atitlán': '#FF8C42'})
    st.plotly_chart(fig4, use_container_width=True)

st.subheader("Comparación de Modelos Predictivos")

X = data[['temperatura', 'ph', 'oxigeno', 'profundidad']].copy()
y = data['nivel_riesgo'].copy()

X = X.fillna(X.mean())
y = y.fillna(y.mode()[0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

modelos = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(random_state=42)
}

modelos_seleccionados = st.multiselect("Seleccionar modelos a comparar", list(modelos.keys()), default=list(modelos.keys())[:2])

if modelos_seleccionados:
    resultados = []
    for nombre in modelos_seleccionados:
        modelo = modelos[nombre]
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        resultados.append({'Modelo': nombre, 'Accuracy': acc})

    df_resultados = pd.DataFrame(resultados)

    col5, col6 = st.columns([1, 2])

    with col5:
        st.dataframe(df_resultados, use_container_width=True)

    with col6:
        fig5 = px.bar(df_resultados, x='Modelo', y='Accuracy',
                      color='Modelo', color_discrete_sequence=['#2E5090', '#FF8C42', '#4A7BB7'])
        fig5.update_layout(showlegend=False)
        st.plotly_chart(fig5, use_container_width=True)

Overwriting app.py


In [12]:
!streamlit run app.py &>/content/logs.txt &

In [17]:
!npx localtunnel --port 8501

⠙your url is: https://red-jobs-stand.loca.lt
^C


In [22]:
!pip install -q pyngrok

import os, time
from urllib.parse import urlparse
from pyngrok import ngrok, conf

NGROK_TOKEN = ""   # <- deben sacar un auth token en ngrok y se coloca aqui, luego se abre la url que da ngrok y se puede visualizar

conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(addr="8501", proto="http", bind_tls=True)
public_url = tunnel.public_url if hasattr(tunnel, "public_url") else str(tunnel)
print("Public URL:", public_url)

u = urlparse(public_url)
host = u.hostname
port = 443 if u.scheme == "https" else 80
print("Host:", host, "Port:", port)

cfg_dir = ".streamlit"
os.makedirs(cfg_dir, exist_ok=True)
cfg = f"""
[server]
headless = true
address = "0.0.0.0"
port = 8501
enableCORS = false
enableXsrfProtection = false

[browser]
# host público (sin https://)
serverAddress = "{host}"
serverPort = {port}
"""
with open(os.path.join(cfg_dir, "config.toml"), "w") as f:
    f.write(cfg)

print("Wrote .streamlit/config.toml with public host. Restarting Streamlit...")

os.system("pkill -f streamlit || true")
time.sleep(1)

os.system("nohup streamlit run app.py &> /content/logs.txt &")

print("Streamlit iniciado. Espera unos segundos y abre:", public_url)
print("Puedes ver logs con: tail -n 200 /content/logs.txt")

Public URL: https://nonexaggeratory-nonlitigiously-carissa.ngrok-free.dev
Host: nonexaggeratory-nonlitigiously-carissa.ngrok-free.dev Port: 443
Wrote .streamlit/config.toml with public host. Restarting Streamlit...
Streamlit iniciado. Espera unos segundos y abre: https://nonexaggeratory-nonlitigiously-carissa.ngrok-free.dev
Puedes ver logs con: tail -n 200 /content/logs.txt
